#  ⭐ `dim_forma_farmaceutica`


In [1]:
%run ../../config/bootstrap.py

import pandas as pd
import numpy as np
import re 
from unidecode import unidecode
from rapidfuzz import process, fuzz


from pathlib import Path
from utils import get_project_root, normalizar_forma_farmaceutica_raw, criar_dim_forma_farmaceutica, add_key_value_columns

In [2]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
path = Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos.parquet")  
bronze['FORMA_FARMACEUTICA']= bronze['FORMA_FARMACEUTICA'].fillna('DESCONHECIDO')
bronze = (
    bronze[['FORMA_FARMACEUTICA']]
    .value_counts()
    .rename('FREQUENCIA_REGISTROS')
    .reset_index()
) 
bronze.columns

Index(['FORMA_FARMACEUTICA', 'FREQUENCIA_REGISTROS'], dtype='object')

In [4]:
bronze.to_csv(Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos_FORMA_FARMACEUTICA.csv", sep=";", index=False)

In [5]:
bronze.head(2)

,FORMA_FARMACEUTICA,FREQUENCIA_REGISTROS
0,DESCONHECIDO,400464
1,Tablet,9792


In [6]:
bronze.FORMA_FARMACEUTICA.nunique()

4494

In [7]:
bronze.FORMA_FARMACEUTICA.value_counts().head(20)

FORMA_FARMACEUTICA
DESCONHECIDO                                                    1
Cada dose de 0,5 mL da vacina papilomavírus humano 9-valente    1
Capsula dura de liberação prolongada                            1
Capsula de Liberacao Retardada                                  1
Caplet                                                          1
Carboplatina                                                    1
Caneta pré-preenchida auto injetora                             1
Caneta Pré-preenchida                                           1
Cada 0,5 mL de dose intramuscular é formulada para conter 2,    1
Capsulas                                                        1
Cachet                                                          1
CRORIDRATO DE HIDRALAZINA 50 MG COMPRIMIDO                      1
CREME HIDRATANTE                                                1
CPR                                                             1
CP REV                                                   

# 🥈 Silver

normalized data, medium quality


### dev

In [8]:
silver_dev = bronze.copy()

In [9]:
silver_dev, dim_forma = criar_dim_forma_farmaceutica(silver_dev)

In [10]:
dim_forma

,FORMA_FARMACEUTICA,FORMA_FARMACEUTICA_VALOR
0,Desconhecida,1
1,Comprimido,2
2,Solucao injetavel,3
3,Outros,4
4,Solucao para infusao,5
5,Po para solucao injetavel,6
6,Capsula,7
7,Colirio,8
8,Solucao oral,9
9,Inalacao / spray,10


In [11]:
silver_dev.head()

,FORMA_FARMACEUTICA,FREQUENCIA_REGISTROS,FORMA_FARMACEUTICA_CHAVE,FORMA_FARMACEUTICA_VALOR
0,DESCONHECIDO,400464,Desconhecida,1
1,Tablet,9792,Comprimido,2
2,comprimido,7291,Comprimido,2
3,Solução injetável,7177,Solucao injetavel,3
4,solução injetável,7077,Solucao injetavel,3


agora salvamos esse arquivo para validacao manual e criacao do [canonical](https://docs.google.com/spreadsheets/d/1gN6nNe7hV2dDjs54b2Mb631hEUO5EChy7Y-Hw7bgjXk/edit?gid=663735313#gid=663735313)

In [12]:
silver_dev.to_excel(Path(project_root) / "data/02_silver/hist_dim_forma_farmaceutica/hist_dim_forma_farmaceutica_MANUAL.xlsx",index=False)

### elt

In [24]:
silver = pd.read_excel(Path(project_root) /  "data/02_silver/hist_dim_forma_farmaceutica/hist_dim_forma_farmaceutica_MANUAL.xlsx")
#silver = silver.drop_duplicates(subset=['DETENTOR_REGISTRO'])
silver.head()


,FORMA_FARMACEUTICA,FORMA_FARMACEUTICA_CHAVE
0,Adesivo Transdérmico,Adesivo transdermico
1,adesivo,Adesivo transdermico
2,adesivo transdérmico,Adesivo transdermico
3,Adesivo,Adesivo transdermico
4,Adesivo,Adesivo transdermico


In [25]:
silver.head()

,FORMA_FARMACEUTICA,FORMA_FARMACEUTICA_CHAVE
0,Adesivo Transdérmico,Adesivo transdermico
1,adesivo,Adesivo transdermico
2,adesivo transdérmico,Adesivo transdermico
3,Adesivo,Adesivo transdermico
4,Adesivo,Adesivo transdermico


In [26]:
silver.FORMA_FARMACEUTICA.nunique()

3574

In [27]:
hist = add_key_value_columns(
    silver,
    base_col="FORMA_FARMACEUTICA",
    normalized_col="FORMA_FARMACEUTICA_CHAVE",
)

hist.head()

,FORMA_FARMACEUTICA,FORMA_FARMACEUTICA_CHAVE,FORMA_FARMACEUTICA_VALOR
0,Adesivo Transdérmico,1,Adesivo transdermico
1,adesivo,1,Adesivo transdermico
2,adesivo transdérmico,1,Adesivo transdermico
3,Adesivo,1,Adesivo transdermico
4,Adesivo,1,Adesivo transdermico


In [28]:
dim = hist[['FORMA_FARMACEUTICA_CHAVE','FORMA_FARMACEUTICA_VALOR']].drop_duplicates()
dim.sort_values(by="FORMA_FARMACEUTICA_CHAVE", ascending=True)

,FORMA_FARMACEUTICA_CHAVE,FORMA_FARMACEUTICA_VALOR
0,1,Adesivo transdermico
31,2,Capsula
255,3,Colirio
270,4,Comprimido
701,5,Creme
715,6,Forma vaginal
725,7,Gel
745,8,Gotas orais
782,9,Implante
789,10,Inalacao / spray


In [29]:
hist.to_parquet(Path(project_root) / "data/02_silver/hist_dim_forma_farmaceutica/hist_dim_forma_farmaceutica.parquet", index=False)

# 🥇 Gold


In [30]:
dim.to_parquet(Path(project_root) / "data/03_gold/dim_forma_farmaceutica/dim_forma_farmaceutica.parquet", index=False)

In [31]:
 
dim.to_csv(Path(project_root) / "data/03_gold/dim_forma_farmaceutica/dim_forma_farmaceutica.csv", sep=";", index=False)